In [0]:
%sql
CREATE CATALOG IF NOT EXISTS spotify_etl;
CREATE SCHEMA  IF NOT EXISTS spotify_etl.raw;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS spotify_etl.raw.spotify_tokens (
  token_name    STRING  NOT NULL,
  access_token  STRING,
  refresh_token STRING,
  expires_at    BIGINT,
  updated_at    TIMESTAMP
)
USING DELTA
TBLPROPERTIES (
  delta.enableChangeDataFeed = true,
  delta.columnMapping.mode = 'name'
);

-- Constrângeri
ALTER TABLE spotify_etl.raw.spotify_tokens
  ADD CONSTRAINT tokens_expires_at_nonneg CHECK (expires_at IS NULL OR expires_at >= 0);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS spotify_etl.raw.spotify_api_calls (
  ingestion_ts          TIMESTAMP NOT NULL,
  ingestion_date        DATE      NOT NULL,

  run_id                STRING    NOT NULL,
  token_name            STRING,
  method                STRING    NOT NULL,
  endpoint              STRING,
  request_url           STRING    NOT NULL,

  request_params_json   STRING,
  request_body_json     STRING,

  http_status           INT,
  response_headers_json STRING,
  payload_json          STRING,
  error                STRING,

  attempt               INT,
  request_hash          STRING    NOT NULL
)
USING DELTA
PARTITIONED BY (ingestion_date)
TBLPROPERTIES (
  delta.enableChangeDataFeed = true,
  delta.autoOptimize.optimizeWrite = true,
  delta.autoOptimize.autoCompact = true,
  delta.columnMapping.mode = 'name'
);

-- Constrângeri
ALTER TABLE spotify_etl.raw.spotify_api_calls
  ADD CONSTRAINT api_http_status_range CHECK (http_status IS NULL OR (http_status >= 100 AND http_status <= 599));

ALTER TABLE spotify_etl.raw.spotify_api_calls
  ADD CONSTRAINT api_attempt_positive CHECK (attempt IS NULL OR attempt >= 1);

ALTER TABLE spotify_etl.raw.spotify_api_calls
  ADD CONSTRAINT api_method_allowed CHECK (upper(method) IN ('GET','POST','PUT','PATCH','DELETE'));

In [0]:
%sql
CREATE TABLE IF NOT EXISTS spotify_etl.raw.spotify_api_errors (
  ingestion_ts          TIMESTAMP NOT NULL,
  ingestion_date        DATE      NOT NULL,

  run_id                STRING    NOT NULL,
  token_name            STRING,
  method                STRING    NOT NULL,
  endpoint              STRING,
  request_url           STRING    NOT NULL,

  request_params_json   STRING,
  request_body_json     STRING,

  http_status           INT,
  response_headers_json STRING,
  payload_json          STRING,
  error                STRING,

  attempt               INT,
  request_hash          STRING    NOT NULL
)
USING DELTA
PARTITIONED BY (ingestion_date)
TBLPROPERTIES (
  delta.enableChangeDataFeed = true,
  delta.autoOptimize.optimizeWrite = true,
  delta.autoOptimize.autoCompact = true,
  delta.columnMapping.mode = 'name'
);


In [0]:
from pyspark.sql import functions as F

def write_raw_api_calls(df, table = "spotify_etl.raw.spotify_api_calls"):
    out = (df
        .withColumn("ingestion_ts", F.current_timestamp())
        .withColumn("ingestion_date", F.current_date())
    )
    out.write.format('delta').mode('append').saveAsTable(table)

In [0]:
%sql
-- Rulează periodic (ex. săptămânal) dacă ai multe fișiere mici.
-- OPTIMIZE spotify_etl.raw.spotify_api_calls
-- ZORDER BY (endpoint, run_id, request_hash);